In [38]:
import numpy as np
import pandas as pd
import country_converter as coco

In [25]:
# # Load UN speeches dataset
# speeches = pd.read_parquet("../data/un_speeches.parquet")

# speeches['country_code'] = speeches['country_code'].astype(str)
# speeches['year'] = speeches['year'].astype(int)

In [47]:
cc = coco.CountryConverter()

conflict = pd.read_excel("../data/2026_ucdp-prio-acd-261.xlsx")

def convert_gwno_cell(val):
    """Converts a single GW code (int, float, or comma-separated string) to ISO3."""
    if pd.isna(val) or val == 0 or val == '0':
        return np.nan
    
    # Handle comma-separated lists of GW codes (e.g., "365, 370" -> ["365", "370"])
    if isinstance(val, str) and ',' in val:
        codes = [c.strip() for c in val.split(',') if c.strip() not in ['0', '']]
        converted = [cc.convert(names=c, src='GWcode', to='ISO3', not_found=None) for c in codes]
        # Drop Nones and join with comma (e.g., "RUS, BLR")
        valid = [c for c in converted if c is not None]
        return ', '.join(valid) if valid else np.nan

    # Handle single integer or string code
    try:
        clean_code = int(float(val))
        if clean_code == 0:
            return np.nan
        res = cc.convert(names=clean_code, src='GWcode', to='ISO3', not_found=None)
        return res if res != 'not found' else np.nan
    except (ValueError, TypeError):
        return np.nan

def convert_gwno(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    for col in cols:
        if col in df.columns:
            # Generate target column name (e.g., 'gwno_a' -> 'iso3_a')
            col_suffix = col.replace('gwno_', '')
            iso_col = f'iso3_{col_suffix}'
            
            # Apply cell-by-cell conversion
            df[iso_col] = df[col].apply(convert_gwno_cell)
            
    return df

to_convert = ['gwno_a', 'gwno_a_2nd', 'gwno_b', 'gwno_b_2nd', 'gwno_loc']
conflict = convert_gwno(conflict, to_convert)

conflict

751 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
678 not found in GWcode
345 not found in GWcode
345 not found in GWcode
345 not found in GWcode
678 not found in GWcode
678 not found in GWcode
345 not found in GWcode
345 not found in GWcode
751 not found in GWcode
751 not found in

,conflict_id,location,side_a,side_a_id,side_a_2nd,side_b,side_b_id,side_b_2nd,incompatibility,territory_name,...,gwno_b,gwno_b_2nd,gwno_loc,region,version,iso3_a,iso3_a_2nd,iso3_b,iso3_b_2nd,iso3_loc
0,200,Bolivia,Government of Bolivia,23,NaN,Popular Revolutionary Movement,719,NaN,2,NaN,...,NaN,NaN,145,5,26.1,BOL,NaN,NaN,NaN,BOL
1,200,Bolivia,Government of Bolivia,23,NaN,MNR,720,NaN,2,NaN,...,NaN,NaN,145,5,26.1,BOL,NaN,NaN,NaN,BOL
2,200,Bolivia,Government of Bolivia,23,NaN,MNR,720,NaN,2,NaN,...,NaN,NaN,145,5,26.1,BOL,NaN,NaN,NaN,BOL
3,200,Bolivia,Government of Bolivia,23,NaN,ELN,721,NaN,2,NaN,...,NaN,NaN,145,5,26.1,BOL,NaN,NaN,NaN,BOL
4,201,Cambodia (Kampuchea),Government of France,33,NaN,Khmer Issarak,160,NaN,1,Cambodia,...,NaN,NaN,811,3,26.1,FRA,NaN,NaN,NaN,KHM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2811,16099,"United Kingdom, United States of America, Yeme...","Government of United Kingdom, Government of Un...","28, 3","Government of Australia, Government of Bahrain...",Government of Yemen (North Yemen),123,NaN,1,"Red Sea, Bab al Mandab and Gulf of Aden",...,678,NaN,"2, 200, 678","1, 2, 5",26.1,"GBR, USA","CAN, NLD, NOR, BHR, LKA, SGP, AUS, NZL",678,NaN,"USA, GBR, 678"
2812,16099,"United Kingdom, United States of America, Yeme...","Government of United Kingdom, Government of Un...","28, 3","Government of Australia, Government of Bahrain...",Government of Yemen (North Yemen),123,NaN,1,"Red Sea, Bab al Mandab and Gulf of Aden",...,678,NaN,"2, 200, 678","1, 2, 5",26.1,"GBR, USA","CAN, NLD, NOR, BHR, LKA, SGP, AUS, NZL",678,NaN,"USA, GBR, 678"
2813,16292,Tajikistan,Government of Tajikistan,131,NaN,IS,234,NaN,1,Islamic State,...,NaN,NaN,702,3,26.1,TJK,NaN,NaN,NaN,TJK
2814,16292,Tajikistan,Government of Tajikistan,131,NaN,IS,234,NaN,1,Islamic State,...,NaN,NaN,702,3,26.1,TJK,NaN,NaN,NaN,TJK


In [49]:
# 4. Join with Ideal Point Dyads (if adding ideological distance)
# Load your pre-calculated dyadic ideal points
ideal_dyads = pd.read_csv("../data/IdealPointDyads1946-2025.csv")

# Standardize type matching for the dyadic join
ideal_dyads['year'] = ideal_dyads['year'].astype(int)

# # Merge dyadic ideal points matching Side A and Side B of the UCDP conflict
# final_merged = pd.merge(
#     conflict_dyads,
#     ideal_dyads,
#     left_on=['gwno_a', 'gwno_b', 'year'],
#     right_on=['ccode1', 'ccode2', 'year'],
#     how='left'
# )
conflict[['territory_name', 'year', 'intensity_level', 'cumulative_intensity',
       'type_of_conflict', 'start_date', 'start_prec', 'start_date2',
       'start_prec2', 'ep_end', 'ep_end_date', 'ep_end_prec']]

,territory_name,year,intensity_level,cumulative_intensity,type_of_conflict,start_date,start_prec,start_date2,start_prec2,ep_end,ep_end_date,ep_end_prec
0,NaN,1946,2,1,3,1946-07-18,1,1946-07-21,2,1,1946-07-21,NaN
1,NaN,1949,1,1,3,1946-07-18,1,1949-09-15,3,1,1949-09-15,NaN
2,NaN,1952,1,1,3,1946-07-18,1,1952-04-09,1,1,1952-04-12,NaN
3,NaN,1967,1,1,3,1946-07-18,1,1967-03-31,3,1,1967-10-16,NaN
4,Cambodia,1946,1,0,1,1946-08-31,3,1946-08-31,3,0,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2811,"Red Sea, Bab al Mandab and Gulf of Aden",2024,1,0,2,2023-12-31,1,2024-02-03,1,0,NaT,NaN
2812,"Red Sea, Bab al Mandab and Gulf of Aden",2025,1,0,2,2023-12-31,1,2024-02-03,1,0,NaT,NaN
2813,Islamic State,2018,1,0,3,2018-07-30,1,2018-11-08,2,0,NaT,NaN
2814,Islamic State,2019,1,0,3,2018-07-30,1,2018-11-08,2,1,2019-11-06,NaN


In [28]:
final_merged

,conflict_id,dyad_id,location_inc,side_a,side_a_id,side_a_2nd,side_b,side_b_id,side_b_2nd,incompatibility,...,iso3_b,ccode1,iso3c1,Countryname1,ccode2,iso3c2,Countryname2,IdealPointFP1,IdealPointFP2,AbsIdealDiff
0,205,406,Iran,Government of Iran,114,NaN,KDPI,164,NaN,1,...,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,205,406,Iran,Government of Iran,114,NaN,KDPI,164,NaN,1,...,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,205,406,Iran,Government of Iran,114,NaN,KDPI,164,NaN,1,...,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,205,406,Iran,Government of Iran,114,NaN,KDPI,164,NaN,1,...,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,205,406,Iran,Government of Iran,114,NaN,KDPI,164,NaN,1,...,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2076,309,18621,Sudan,Government of Sudan,112,NaN,SFA,9645,NaN,2,...,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2077,234,18806,Israel,Government of Israel,121,NaN,Palestinian Mujahideen Movement,9806,NaN,1,...,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2078,234,18806,Israel,Government of Israel,121,NaN,Palestinian Mujahideen Movement,9806,NaN,1,...,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2079,234,18806,Israel,Government of Israel,121,NaN,Palestinian Mujahideen Movement,9806,NaN,1,...,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


KeyError: "['gwno_battleiso3_a'] not in index"

,conflict_id,dyad_id,location_inc,side_a,side_a_id,side_a_2nd,side_b,side_b_id,side_b_2nd,incompatibility,...,gwno_a,gwno_a_2nd,gwno_b,gwno_b_2nd,gwno_loc,gwno_battle,region,version,iso3_a,iso3_b
0,205,406,Iran,Government of Iran,114,NaN,KDPI,164,NaN,1,...,630,NaN,NaN,NaN,630,630,2,26.1,IRN,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
1,205,406,Iran,Government of Iran,114,NaN,KDPI,164,NaN,1,...,630,NaN,NaN,NaN,630,"630, 645",2,26.1,IRN,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
2,205,406,Iran,Government of Iran,114,NaN,KDPI,164,NaN,1,...,630,NaN,NaN,NaN,630,"630, 645",2,26.1,IRN,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
3,205,406,Iran,Government of Iran,114,NaN,KDPI,164,NaN,1,...,630,NaN,NaN,NaN,630,"630, 645",2,26.1,IRN,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
4,205,406,Iran,Government of Iran,114,NaN,KDPI,164,NaN,1,...,630,NaN,NaN,NaN,630,"630, 645",2,26.1,IRN,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2076,309,18621,Sudan,Government of Sudan,112,NaN,SFA,9645,NaN,2,...,625,NaN,NaN,NaN,625,625,4,26.1,SDN,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
2077,234,18806,Israel,Government of Israel,121,NaN,Palestinian Mujahideen Movement,9806,NaN,1,...,666,NaN,NaN,NaN,666,666,2,26.1,ISR,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
2078,234,18806,Israel,Government of Israel,121,NaN,Palestinian Mujahideen Movement,9806,NaN,1,...,666,NaN,NaN,NaN,666,666,2,26.1,ISR,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
2079,234,18806,Israel,Government of Israel,121,NaN,Palestinian Mujahideen Movement,9806,NaN,1,...,666,NaN,NaN,NaN,666,666,2,26.1,ISR,"[ALA, ASM, AIA, ATA, ABW, BMU, BES, BVT, IOT, ..."
